# 🎤 NotyCaption Pro - Vocal Enhancement

Extract vocals using Spleeter AI on **T4 GPU**.


In [ ]:
# Check GPU
!nvidia-smi
import torch
print('CUDA Available:', torch.cuda.is_available())


In [ ]:
# Install dependencies
!apt-get update -qq && apt-get install -y ffmpeg -qq
!pip install -q spleeter google-auth google-auth-oauthlib google-api-python-client
print('✅ Installed')


In [ ]:
# Google Drive Auth
from google.colab import auth
auth.authenticate_user()

from google.auth import default
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload, MediaFileUpload
import io, json, os

creds, _ = default()
drive_service = build('drive', 'v3', credentials=creds)
print('✅ Drive Ready')


In [ ]:
# Audio Configuration
AUDIO_ID = '{{AUDIO_ID}}'
OPERATION_ID = '{{OPERATION_ID}}'
AUDIO_NAME = '{{AUDIO_NAME}}'

print(f'📁 Audio ID: {AUDIO_ID}')
print(f'📄 Output: {AUDIO_NAME}_vocals.wav')


In [ ]:
# Download Audio
def download_file(file_id, destination):
    request = drive_service.files().get_media(fileId=file_id)
    fh = io.FileIO(destination, 'wb')
    downloader = MediaIoBaseDownload(fh, request)
    done = False
    while not done:
        status, done = downloader.next_chunk()
        if status:
            print(int(status.progress()*100), '%')

audio_path = f'/content/audio_{OPERATION_ID}.wav'
download_file(AUDIO_ID, audio_path)
print('✅ Audio Downloaded')


In [ ]:
# Spleeter Vocal Extraction
from spleeter.separator import Separator

print('Loading Spleeter...')
separator = Separator('spleeter:2stems')

print('Separating vocals...')
separator.separate_to_file(audio_path, '/content/output')

print('✅ Vocals Extracted')


In [ ]:
# Upload Enhanced Audio
vocals_path = f'/content/output/audio_{OPERATION_ID}/vocals.wav'
if not os.path.exists(vocals_path):
    vocals_path = '/content/output/vocals.wav'

output_name = f'{AUDIO_NAME}_vocals.wav'

file_metadata = {'name': output_name}
media = MediaFileUpload(vocals_path, mimetype='audio/wav', resumable=True)
uploaded = drive_service.files().create(body=file_metadata, media_body=media, fields='id').execute()

print(f'✅ Uploaded: {uploaded["id"]}')

# Send result back
result = {
    'success': True,
    'file_id': uploaded['id'],
    'file_name': output_name,
    'operation_id': OPERATION_ID
}

from IPython.display import display, Javascript
display(Javascript(f"sessionStorage.setItem('colab_result_{OPERATION_ID}', '{json.dumps(result)}'); console.log('✅ Result saved');"))

print('🎉 All done! Close this tab.')
